In [39]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [40]:
df = pd.read_csv("../clean_data/train_data_features.csv")

In [41]:
# Binary classification: predict 90th percentile prices for HB_NORTH

# Set index
if 'interval_start_local' in df.columns:
    df['interval_start_local'] = pd.to_datetime(df['interval_start_local'])
    df.set_index('interval_start_local', inplace=True)
    df.sort_index(inplace=True)

# Define 90th percentile threshold for HB_NORTH
threshold_90 = df['HB_NORTH'].quantile(0.90)
print(f'90th percentile threshold for HB_NORTH: {threshold_90:.2f}')

# Create binary target: 1 if >= 90th percentile, 0 otherwise
df['is_90th_percentile'] = (df['HB_NORTH'] >= threshold_90).astype(int)
print(f'Class distribution: {df["is_90th_percentile"].value_counts().to_dict()}')

# Select price features (all hub columns)
price_cols = ['price_ramp_hist', 'price_rolling_6h_hist', 'price_rolling_24h_hist', 'price_vol_24h_hist', 
              'price_lag1', 'price_lag2', 'price_lag4', 'price_lag12', 'price_lag24',
              'load_ramp_hist', 'load_rolling_6h_hist', 'load_rolling_24h_hist', 'load_vol_24h_hist', 'load_system_lag1', 
              'load_system_lag2', 'load_system_lag4', 'load_system_lag12', 'load_system_lag24']


# Build feature matrix X (price columns only) and binary target y
X = df[price_cols]
y = df['is_90th_percentile']

# Drop rows with NaNs
data = pd.concat([X, y], axis=1).dropna()
X = data[price_cols]
y = data['is_90th_percentile']

# Time-based train/test split (first 80% train, last 20% test)
split_idx = int(len(data) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit logistic regression with increased max_iter
model = LogisticRegression(max_iter=5000, random_state=42)
model.fit(X_train_scaled, y_train)

# Predict probabilities
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
y_pred = model.predict(X_test_scaled)

# Calculate AUC
auc = roc_auc_score(y_test, y_pred_proba)

print(f'\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'AUC: {auc:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred))

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(f'\nConfusion Matrix:')
print(cm)
tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
print(f'TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}')

# Explicit precision and recall for class 1
precision_class_1 = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_class_1 = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f'Precision (class 1): {precision_class_1:.4f}')
print(f'Recall (class 1): {recall_class_1:.4f}')

90th percentile threshold for HB_NORTH: 42.42
Class distribution: {0: 15625, 1: 1737}

Train shape: (13889, 18), Test shape: (3473, 18)
AUC: 0.9119

Classification Report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      3193
           1       0.83      0.25      0.38       280

    accuracy                           0.94      3473
   macro avg       0.88      0.62      0.67      3473
weighted avg       0.93      0.94      0.92      3473


Confusion Matrix:
[[3179   14]
 [ 211   69]]
TN: 3179, FP: 14, FN: 211, TP: 69
Precision (class 1): 0.8313
Recall (class 1): 0.2464


In [42]:
# Adjust classification cutoff (default is 0.5)
cutoff = 0.3  # Change this value to lower/raise the threshold
y_pred_adjusted = (y_pred_proba >= cutoff).astype(int)

# Recalculate metrics with new cutoff
from sklearn.metrics import confusion_matrix
cm_adj = confusion_matrix(y_test, y_pred_adjusted)
tn_adj, fp_adj, fn_adj, tp_adj = cm_adj[0,0], cm_adj[0,1], cm_adj[1,0], cm_adj[1,1]

precision_adj = tp_adj / (tp_adj + fp_adj) if (tp_adj + fp_adj) > 0 else 0
recall_adj = tp_adj / (tp_adj + fn_adj) if (tp_adj + fn_adj) > 0 else 0

print(f'\nWith cutoff = {cutoff}:')
print(f'Precision (class 1): {precision_adj:.4f}')
print(f'Recall (class 1): {recall_adj:.4f}')
print(f'Confusion Matrix: TN={tn_adj}, FP={fp_adj}, FN={fn_adj}, TP={tp_adj}')


With cutoff = 0.3:
Precision (class 1): 0.6458
Recall (class 1): 0.4429
Confusion Matrix: TN=3125, FP=68, FN=156, TP=124


In [43]:
# Linear regression to predict HB_NORTH price (with feature standardization and capping extreme values)


# Reload data and prepare features
df_lr = df.copy()

# Set index
if 'interval_start_local' in df_lr.columns:
    df_lr['interval_start_local'] = pd.to_datetime(df_lr['interval_start_local'])
    df_lr.set_index('interval_start_local', inplace=True)
    df_lr.sort_index(inplace=True)

# Cap HB_NORTH values above the 99th percentile to 250
#threshold_99_lr = df_lr['HB_NORTH'].quantile(0.99)
#cap_value = 250
#df_lr['HB_NORTH'] = np.where(df_lr['HB_NORTH'] > threshold_99_lr, cap_value, df_lr['HB_NORTH'])

# Remove the is_90th_percentile column and use all other numeric features as predictors
X_lr = df_lr.drop(columns=['HB_NORTH', 'is_90th_percentile'])
X_lr = X_lr.select_dtypes(include=[np.number])
y_lr = df_lr['HB_NORTH']

# Drop rows with NaNs
data_lr = pd.concat([X_lr, y_lr], axis=1).dropna()
X_lr = data_lr.drop(columns=['HB_NORTH'])
y_lr = data_lr['HB_NORTH']

# Time-based train/test split (first 80% train, last 20% test)
split_idx_lr = int(len(data_lr) * 0.8)
X_train_lr, X_test_lr = X_lr.iloc[:split_idx_lr], X_lr.iloc[split_idx_lr:]
y_train_lr, y_test_lr = y_lr.iloc[:split_idx_lr], y_lr.iloc[split_idx_lr:]

# Standardize features
scaler_lr = StandardScaler()
X_train_lr_scaled = scaler_lr.fit_transform(X_train_lr)
X_test_lr_scaled = scaler_lr.transform(X_test_lr)

# Fit linear regression
model_lr = LinearRegression()
model_lr.fit(X_train_lr_scaled, y_train_lr)
y_pred_lr = model_lr.predict(X_test_lr_scaled)

# Naive baseline: predict price from 24 hours before
y_baseline_lr = y_lr.iloc[split_idx_lr-24:split_idx_lr+len(y_test_lr)-24].values

# Calculate performance metrics
rmse_lr = np.sqrt(mean_squared_error(y_test_lr, y_pred_lr))
mae_lr = mean_absolute_error(y_test_lr, y_pred_lr)
r2_lr = r2_score(y_test_lr, y_pred_lr)

rmse_baseline_lr = np.sqrt(mean_squared_error(y_test_lr, y_baseline_lr))
mae_baseline_lr = mean_absolute_error(y_test_lr, y_baseline_lr)

improvement_rmse = (rmse_baseline_lr - rmse_lr) / rmse_baseline_lr * 100
improvement_mae = (mae_baseline_lr - mae_lr) / mae_baseline_lr * 100

print(f'Linear Regression - Predicting HB_NORTH Price (Standardized Features, Capped >99th to {cap_value})')
print(f'=' * 50)
print(f'\nTrain shape: {X_train_lr.shape}, Test shape: {X_test_lr.shape}')
print(f'Number of features: {X_train_lr.shape[1]}')
print(f'\nModel Performance:')
print(f'  RMSE: {rmse_lr:.4f}')
print(f'  MAE:  {mae_lr:.4f}')
print(f'  R2:   {r2_lr:.4f}')
print(f'\nNaive Baseline (24-hour lag) Performance:')
print(f'  RMSE: {rmse_baseline_lr:.4f}')
print(f'  MAE:  {mae_baseline_lr:.4f}')
print(f'\nImprovement over baseline:')
print(f'  RMSE improvement: {improvement_rmse:.2f}%')
print(f'  MAE improvement:  {improvement_mae:.2f}%')

Linear Regression - Predicting HB_NORTH Price (Standardized Features, Capped >99th to 250)

Train shape: (13889, 128), Test shape: (3473, 128)
Number of features: 128

Model Performance:
  RMSE: 118.1700
  MAE:  27.3241
  R2:   -0.0148

Naive Baseline (24-hour lag) Performance:
  RMSE: 157.6853
  MAE:  18.5570

Improvement over baseline:
  RMSE improvement: 25.06%
  MAE improvement:  -47.24%


In [29]:
# Linear regression on ln(HB_NORTH) with back-transformed metrics and naive baseline comparison
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Reload data and prepare features
log_df = df.copy()

# Set index
if 'interval_start_local' in log_df.columns:
    log_df['interval_start_local'] = pd.to_datetime(log_df['interval_start_local'])
    log_df.set_index('interval_start_local', inplace=True)
    log_df.sort_index(inplace=True)

# Remove the is_90th_percentile column and use all other numeric features as predictors
X_log = log_df.drop(columns=['HB_NORTH', 'is_90th_percentile'])
X_log = X_log.select_dtypes(include=[np.number])
y_log_raw = log_df['HB_NORTH']

# Keep only strictly positive targets for natural log
mask_positive = y_log_raw > 0
X_log = X_log[mask_positive]
y_log_raw = y_log_raw[mask_positive]

# Log-transform target
y_log = np.log(y_log_raw)

# Align and drop NaNs
data_log = pd.concat([X_log, y_log], axis=1).dropna()
X_log = data_log.iloc[:, :-1]
y_log = data_log.iloc[:, -1]
y_raw_aligned = y_log_raw.loc[data_log.index]

# Time-based train/test split (first 80% train, last 20% test)
split_idx_log = int(len(data_log) * 0.8)
X_train_log, X_test_log = X_log.iloc[:split_idx_log], X_log.iloc[split_idx_log:]
y_train_log, y_test_log = y_log.iloc[:split_idx_log], y_log.iloc[split_idx_log:]
y_raw_train, y_raw_test = y_raw_aligned.iloc[:split_idx_log], y_raw_aligned.iloc[split_idx_log:]

# Standardize features
scaler_log = StandardScaler()
X_train_log_scaled = scaler_log.fit_transform(X_train_log)
X_test_log_scaled = scaler_log.transform(X_test_log)

# Fit linear regression on log target
model_log = LinearRegression()
model_log.fit(X_train_log_scaled, y_train_log)
y_pred_log = model_log.predict(X_test_log_scaled)

# Back-transform predictions to original scale
y_pred_orig = np.exp(y_pred_log)
y_test_orig = np.exp(y_test_log)

# Naive baseline: predict price from 24 hours before (original scale)
y_baseline_orig = y_raw_aligned.iloc[split_idx_log-24:split_idx_log+len(y_test_log)-24].values

# Calculate performance metrics on original scale
rmse_log = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
mae_log = mean_absolute_error(y_test_orig, y_pred_orig)
r2_log = r2_score(y_test_orig, y_pred_orig)

rmse_baseline_log = np.sqrt(mean_squared_error(y_test_orig, y_baseline_orig))
mae_baseline_log = mean_absolute_error(y_test_orig, y_baseline_orig)

improvement_rmse_log = (rmse_baseline_log - rmse_log) / rmse_baseline_log * 100
improvement_mae_log = (mae_baseline_log - mae_log) / mae_baseline_log * 100

print('Linear Regression on ln(HB_NORTH)')
print('=' * 50)
print(f"Train shape: {X_train_log.shape}, Test shape: {X_test_log.shape}")
print(f"Number of features: {X_train_log.shape[1]}")
print('\nModel Performance (back-transformed to original scale):')
print(f"  RMSE: {rmse_log:.4f}")
print(f"  MAE:  {mae_log:.4f}")
print(f"  R2:   {r2_log:.4f}")
print('\nNaive Baseline (24-hour lag) Performance:')
print(f"  RMSE: {rmse_baseline_log:.4f}")
print(f"  MAE:  {mae_baseline_log:.4f}")
print('\nImprovement over baseline:')
print(f"  RMSE improvement: {improvement_rmse_log:.2f}%")
print(f"  MAE improvement:  {improvement_mae_log:.2f}%")

Linear Regression on ln(HB_NORTH)
Train shape: (13573, 128), Test shape: (3394, 128)
Number of features: 128

Model Performance (back-transformed to original scale):
  RMSE: 139.5773
  MAE:  14.7718
  R2:   -0.3850

Naive Baseline (24-hour lag) Performance:
  RMSE: 167.9711
  MAE:  22.3874

Improvement over baseline:
  RMSE improvement: 16.90%
  MAE improvement:  34.02%


In [22]:
# XGBoost regression to predict HB_NORTH price
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Prepare data (same as linear regression)
X_xgb = X_lr.copy()
y_xgb = y_lr.copy()

# Time-based train/test split (first 80% train, last 20% test)
split_idx_xgb = int(len(X_xgb) * 0.8)
X_train_xgb, X_test_xgb = X_xgb.iloc[:split_idx_xgb], X_xgb.iloc[split_idx_xgb:]
y_train_xgb, y_test_xgb = y_xgb.iloc[:split_idx_xgb], y_xgb.iloc[split_idx_xgb:]

# Optional: Standardize features (XGBoost is tree-based and does not require scaling, but can help with convergence)
# from sklearn.preprocessing import StandardScaler
# scaler_xgb = StandardScaler()
# X_train_xgb = scaler_xgb.fit_transform(X_train_xgb)
# X_test_xgb = scaler_xgb.transform(X_test_xgb)

# Fit XGBoost regressor
model_xgb = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
model_xgb.fit(X_train_xgb, y_train_xgb)
y_pred_xgb = model_xgb.predict(X_test_xgb)

# Calculate performance metrics
rmse_xgb = mean_squared_error(y_test_xgb, y_pred_xgb, squared=False)
mae_xgb = mean_absolute_error(y_test_xgb, y_pred_xgb)
r2_xgb = r2_score(y_test_xgb, y_pred_xgb)

print(f'XGBoost Regression - Predicting HB_NORTH Price')
print('=' * 50)
print(f'\nTrain shape: {X_train_xgb.shape}, Test shape: {X_test_xgb.shape}')
print(f'Number of features: {X_train_xgb.shape[1]}')
print(f'\nModel Performance:')
print(f'  RMSE: {rmse_xgb:.4f}')
print(f'  MAE:  {mae_xgb:.4f}')
print(f'  R2:   {r2_xgb:.4f}')

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/opt/miniconda3/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <636BF463-1886-392D-B8B3-6011C44DCEE9> /opt/miniconda3/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/mini

In [23]:
# 99th percentile of HB_NORTH
if 'HB_NORTH' not in df.columns:
    raise KeyError('HB_NORTH column not found in df')
threshold_99 = df['HB_NORTH'].quantile(0.99)
print(f'99th percentile for HB_NORTH: {threshold_99:.2f}')

99th percentile for HB_NORTH: 248.68


In [25]:
# Rows at/above the 99th percentile for HB_NORTH
if 'HB_NORTH' not in df.columns:
    raise KeyError('HB_NORTH column not found in df')

# Recompute threshold if not in scope
threshold_99 = threshold_99 if 'threshold_99' in locals() else df['HB_NORTH'].quantile(0.99)

rows_above_99 = df[df['HB_NORTH'] >= threshold_99]
print(f'Rows at/above 99th percentile: {len(rows_above_99)}')
rows_above_99.to_csv('../clean_data/train_99th_percentile_HB_NORTH.csv')

Rows at/above 99th percentile: 174
